# notebook

> The in-memory notebook model: a typed `Cell`, the `Notebook` that holds a stack of them (with selection, clipboard, and app-level UI state), and the single running `nb` instance. Pure data -- no rendering, no HTTP, no kernel. `cells.py` (rendering/routes) and `serialize.py` (the .ipynb boundary) both build on this; it depends on nothing of theirs.

In [ ]:
#| default_exp notebook

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Any
from datetime import datetime
from boopiter.llms import DEFAULT_TOOL_SELECTION  # tool-source defaults for a fresh notebook's tool_selection

In [ ]:
#| export
CTYPES = ('code','note','prompt','raw')  # types you can author; 'assistant' is generated

In [ ]:
#| export
class Cell:
    "One notebook cell: its type, source text, any output, and a few UI/export-related flags."
    def __init__(self, id:int, ctype:str, source:str, output:Any=None, visible:bool=True,
                 model:str|None=None, nb_id:str|None=None, export:bool=False, details:str|None=None):
        "Create a cell with the given type, source text, and optional output/visibility/model/id/export state."
        self.id,self.ctype,self.source = id,ctype,source
        self.output,self.visible = output,visible
        self.model = model  # which LLM produced this (assistant cells only)
        self.nb_id = nb_id  # the .ipynb file's own cell id (nbformat), preserved across saves so git diffs stay clean
        self.export = export  # nbdev '#| export' -- kept as a flag, not embedded text; see save_notebook/load_notebook
        self.details = details  # assistant cells only: raw HTML <details> block (model/tokens/reasoning) shown above the reply -- kept OUT of source so it never pollutes llm_context()
        self.ts = datetime.now().strftime('%I:%M:%S %p')

In [ ]:
#| export
class Notebook:
    "The whole in-memory notebook: its cells, selection, clipboard, and app-level UI state."
    def __init__(self):
        "Start a fresh, empty, untitled notebook."
        self.cells, self._nid, self.compose_type, self.selected = [], 0, 'code', None
        self.name = 'untitled'
        self.models = []  # every locally-available LLM (info dicts, see get_model_list())
        self.standard_model, self.reasoning_model = None, None  # 'id' strings (see get_ollama_list()) picked in the brain-icon menu's two dropdowns
        self.use_reasoning = False  # brain-icon toggle -- which of the two above actually answers Prompt cells; see active_model()
        self.reasoning_effort = 'm'  # 'l'/'m'/'h' -- only applies while use_reasoning is on; passed through as Chat(...)(think=...), see stream_llm_reply()
        self.clipboard = []  # cell snapshots (plain dicts, not live Cells) for cut/copy/paste
        self.tools = []  # functions the LLM may call on Prompt-cell runs -- see add_tool()
        self.tool_selection = dict(DEFAULT_TOOL_SELECTION)  # which tool sources are active -- toggled by the wrench-icon Tools menu; see get_tool_list()

    def active_model(self) -> str|None:
        "The model 'id' actually in play for Prompt-cell answers: reasoning_model if the brain-icon toggle is on and one's selected, else standard_model."
        return self.reasoning_model if (self.use_reasoning and self.reasoning_model) else self.standard_model

    def insert_at(self, pos:int, ctype:str, source:str, output:Any=None, visible:bool=True,
                  model:str|None=None, nb_id:str|None=None, export:bool=False, details:str|None=None) -> Cell:
        "Create a new cell of type `ctype` and insert it at list index `pos`."
        self._nid += 1
        c = Cell(self._nid, ctype, source, output, visible, model, nb_id, export, details)
        self.cells.insert(pos, c)
        return c

    def add(self, ctype:str, source:str, output:Any=None, visible:bool=True,
            model:str|None=None, nb_id:str|None=None, export:bool=False, details:str|None=None) -> Cell:
        "Create a new cell of type `ctype` and append it to the end of the notebook."
        return self.insert_at(len(self.cells), ctype, source, output, visible, model, nb_id, export, details)

    def index(self, id:int) -> int|None:
        "The list index of the cell with this `id`, or None if it's not present."
        return next((i for i,c in enumerate(self.cells) if c.id==id), None)

    def get(self, id:int) -> Cell|None:
        "The cell with this `id`, or None if it's not present."
        i = self.index(id)
        return self.cells[i] if i is not None else None

    def sel_index(self) -> int|None:
        "The list index of the currently-selected cell, or None if nothing is selected."
        return None if self.selected is None else self.index(self.selected)

    def pair_range(self, id:int) -> tuple[int,int]|None:
        "Indices (start,end) spanning the Prompt+Assistant pair containing `id`, or just (i,i) for a lone cell."
        i = self.index(id)
        if i is None: return None
        c = self.cells[i]
        if c.ctype == 'prompt' and i+1 < len(self.cells) and self.cells[i+1].ctype == 'assistant':
            return (i, i+1)
        if c.ctype == 'assistant' and i-1 >= 0 and self.cells[i-1].ctype == 'prompt':
            return (i-1, i)
        return (i, i)

    def remove(self, id:int) -> None:
        "Delete the cell, or its whole Prompt+Assistant pair if it's part of one."
        rng = self.pair_range(id)
        if rng is None: return
        lo, hi = rng
        del self.cells[lo:hi+1]

    def move(self, id:int, delta:int) -> None:
        "Move the cell (or its Prompt+Assistant pair) up/down as a block, swapping with whatever cell/pair is adjacent."
        rng = self.pair_range(id)
        if rng is None: return
        lo, hi = rng
        if delta < 0:
            if lo == 0: return
            nlo, _ = self.pair_range(self.cells[lo-1].id)
            block, neighbor = self.cells[lo:hi+1], self.cells[nlo:lo]
            self.cells[nlo:hi+1] = block + neighbor
        elif delta > 0:
            if hi >= len(self.cells) - 1: return
            _, nhi = self.pair_range(self.cells[hi+1].id)
            block, neighbor = self.cells[lo:hi+1], self.cells[hi+1:nhi+1]
            self.cells[lo:nhi+1] = neighbor + block

    def _snapshot(self, lo:int, hi:int) -> list[dict]:
        "Plain-dict copies of cells[lo:hi+1], for the clipboard."
        return [{'ctype':c.ctype,'source':c.source,'output':c.output,'visible':c.visible,'model':c.model,
                 'export':c.export,'details':c.details}
                for c in self.cells[lo:hi+1]]

    def copy_range(self, id:int) -> None:
        "Copy the cell (or its pair) into the clipboard, without removing it."
        rng = self.pair_range(id)
        if rng is None: return
        lo, hi = rng
        self.clipboard = self._snapshot(lo, hi)

    def cut_range(self, id:int) -> None:
        "Copy the cell (or its pair) into the clipboard, then remove it."
        rng = self.pair_range(id)
        if rng is None: return
        lo, hi = rng
        self.clipboard = self._snapshot(lo, hi)
        del self.cells[lo:hi+1]
        self.selected = self.cells[min(lo, len(self.cells)-1)].id if self.cells else None

    def paste_after(self, id:int|None) -> list[Cell]:
        "Paste the clipboard as new cells (fresh ids) right after `id`'s pair, or at the end if id is None."
        if not self.clipboard: return []
        rng = self.pair_range(id) if id is not None else None
        pos = rng[1] + 1 if rng else len(self.cells)
        pasted = []
        for snap in self.clipboard:
            pasted.append(self.insert_at(pos, snap['ctype'], snap['source'], snap['output'], snap['visible'],
                                          snap['model'], export=snap.get('export', False), details=snap.get('details')))
            pos += 1
        self.selected = pasted[0].id
        return pasted

    def reset(self) -> None:
        "Discard all cells and start over, as if boopiter had just launched fresh."
        self.cells.clear()
        self._nid, self.selected, self.name = 0, None, 'untitled'

In [ ]:
#| export
nb = Notebook()  # the single running notebook instance

In [ ]:
# pair_range: a lone cell is its own (i,i) range; a Prompt immediately followed by its Assistant
# reply is a (prompt_i, assistant_i) pair, addressable from either side; a Prompt with no reply yet
# (or not immediately followed by one) is just a lone cell.
nb.reset()
raw = nb.add('raw', 'r')
p = nb.add('prompt', 'q'); a = nb.add('assistant', 'ans')
lone_prompt = nb.add('prompt', 'q2')  # no assistant reply -- not a pair

assert nb.pair_range(raw.id) == (nb.index(raw.id), nb.index(raw.id))
assert nb.pair_range(p.id) == (nb.index(p.id), nb.index(a.id)), "pair addressed from the prompt side"
assert nb.pair_range(a.id) == (nb.index(p.id), nb.index(a.id)), "pair addressed from the assistant side"
i = nb.index(lone_prompt.id)
assert nb.pair_range(lone_prompt.id) == (i, i), "a prompt with no reply yet is not a pair"
assert nb.pair_range(99999) is None, "an id that doesn't exist returns None"
nb.reset()

# move: a Prompt+Assistant pair moves as one block; boundary moves (past the start/end) are no-ops.
nb.reset()
first = nb.add('code', '1')
p = nb.add('prompt', 'q'); a = nb.add('assistant', 'ans')
last = nb.add('code', '2')

nb.move(p.id, -1)  # move the pair up, swapping with `first`
assert [c.ctype for c in nb.cells] == ['prompt', 'assistant', 'code', 'code']
assert nb.cells[0].id == p.id and nb.cells[1].id == a.id, "the pair moved as one block, in order"

nb.move(nb.cells[0].id, -1)  # already first -- no-op
assert [c.ctype for c in nb.cells] == ['prompt', 'assistant', 'code', 'code']

nb.move(nb.cells[-1].id, 1)  # already last -- no-op
assert [c.ctype for c in nb.cells] == ['prompt', 'assistant', 'code', 'code']

nb.move(p.id, 1)  # move the pair back down past `first`
assert [c.ctype for c in nb.cells] == ['code', 'prompt', 'assistant', 'code']
nb.reset()
print('pair_range + move: lone cells, pairs, and boundary no-ops all verified')

In [ ]:
# copy/cut/paste: copy leaves the original untouched; cut removes it; paste creates fresh cells
# (new ids) rather than reusing the originals, and inserts them after the given anchor (or at the
# end if no anchor is given).
nb.reset()
a = nb.add('code', 'a'); b = nb.add('code', 'b'); c = nb.add('code', 'c')

nb.copy_range(b.id)
assert len(nb.cells) == 3, "copy must not remove anything"
assert nb.clipboard[0]['source'] == 'b'

pasted = nb.paste_after(a.id)
assert [x.source for x in nb.cells] == ['a', 'b', 'b', 'c'], "pasted copy lands right after the anchor"
assert pasted[0].id != b.id, "a pasted cell is a fresh copy, not the original cell object/id"

nb.reset()
a = nb.add('code', 'a'); b = nb.add('code', 'b'); c = nb.add('code', 'c')
nb.cut_range(b.id)
assert [x.source for x in nb.cells] == ['a', 'c'], "cut removes the cell"
assert nb.clipboard[0]['source'] == 'b', "...but keeps it on the clipboard"
assert nb.selected == nb.cells[min(1, len(nb.cells)-1)].id, "selection lands on a sensible neighbor after cutting"

pasted = nb.paste_after(None)  # no anchor -- appends at the end
assert [x.source for x in nb.cells] == ['a', 'c', 'b']

# cut/paste also carries a Prompt+Assistant pair as one unit.
nb.reset()
lead = nb.add('code', 'lead')
p = nb.add('prompt', 'q'); asst = nb.add('assistant', 'ans')
nb.cut_range(p.id)
assert [x.ctype for x in nb.cells] == ['code'], "cutting a pair removes both cells"
assert len(nb.clipboard) == 2 and nb.clipboard[0]['ctype'] == 'prompt' and nb.clipboard[1]['ctype'] == 'assistant'
nb.paste_after(lead.id)
assert [x.ctype for x in nb.cells] == ['code', 'prompt', 'assistant'], "pasting restores the pair intact, in order"
nb.reset()
print('copy/cut/paste verified, including Prompt+Assistant pairs moving as one unit')

In [ ]:
# remove() deletes a lone cell, or a whole Prompt+Assistant pair when the id addresses one --
# same pairing rule as pair_range/move/cut, just without going through the clipboard.
nb.reset()
a = nb.add('code', 'a'); b = nb.add('code', 'b')
nb.remove(a.id)
assert [x.source for x in nb.cells] == ['b']

nb.reset()
lead = nb.add('code', 'lead')
p = nb.add('prompt', 'q'); asst = nb.add('assistant', 'ans')
trail = nb.add('code', 'trail')
nb.remove(asst.id)  # addressed from the assistant side -- should still take the whole pair
assert [x.ctype for x in nb.cells] == ['code', 'code'], "removing via the assistant side removes its paired prompt too"

nb.reset()
print('remove() verified for lone cells and Prompt+Assistant pairs')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()